## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-10-21-52-00 +0000,wsj,Senior U.S. Officer Is Arrested in U.K. Over A...,https://www.wsj.com/world/europe/member-us-arm...
1,2026-11-10-21-49-03 +0000,nypost,Hochul budget chief swats down tax hike push f...,https://nypost.com/2026/11/10/us-news/hochul-b...
2,2026-11-10-21-48-34 +0000,nypost,Why it’s taking so long to count midterm Elect...,https://nypost.com/2026/11/10/us-news/why-its-...
3,2026-11-10-21-48-00 +0000,wsj,Opinion | The GOP Got Trounced. Now What?,https://www.wsj.com/opinion/the-gop-got-trounc...
4,2026-11-10-21-47-25 +0000,nyt,"Firing of Voice of America Chief Was Illegal, ...",https://www.nytimes.com/2026/11/10/us/politics...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2424/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
114,nvidia,37
155,trump,36
117,huang,27
116,jensen,22
73,house,21


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
219,2026-11-10-15-45-06 +0000,nyt,Nvidia CEO Jensen Huang Told Trump He Did Not ...,https://www.nytimes.com/2026/11/09/us/politics...,215
286,2026-11-10-11-59-50 +0000,nypost,FBI is ‘looking into’ Nvidia boss Jensen Huang...,https://nypost.com/2026/11/10/us-news/fbi-is-l...,189
115,2026-11-10-19-21-16 +0000,nypost,Nvidia loses nearly $450B in two-day wipeout a...,https://nypost.com/2026/11/10/business/nvidia-...,178
429,2026-11-09-23-10-47 +0000,wapo,Trump says Nvidia CEO denied knowing woman he ...,https://www.washingtonpost.com/technology/2026...,174
411,2026-11-10-00-07-08 +0000,nypost,"Nvidia, CEO Jensen Huang hid affair to suspect...",https://nypost.com/2026/11/09/business/nvidia-...,166


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
219,215,2026-11-10-15-45-06 +0000,nyt,Nvidia CEO Jensen Huang Told Trump He Did Not ...,https://www.nytimes.com/2026/11/09/us/politics...
60,121,2026-11-10-20-47-00 +0000,wsj,"House Republicans elected Rep. Jim Jordan (R.,...",https://www.wsj.com/politics/jim-jordan-wins-s...
217,92,2026-11-10-15-47-00 +0000,wsj,Defense Secretary Pete Hegseth gave a Pentagon...,https://www.wsj.com/politics/national-security...
380,75,2026-11-10-03-28-00 +0000,wsj,California voters rejected a proposed billiona...,https://www.wsj.com/politics/elections/califor...
244,62,2026-11-10-14-20-09 +0000,cbc,Anthropic eyes $1 trillion US valuation in hot...,https://www.cbc.ca/news/business/anthropic-ipo...
168,62,2026-11-10-17-36-51 +0000,nypost,Gas prices drop below $4 a gallon for first ti...,https://nypost.com/2026/11/10/business/gas-pri...
319,57,2026-11-10-10-48-00 +0000,nyt,Here’s why last year’s government shutdown wil...,https://www.nytimes.com/live/2026/11/10/busine...
95,49,2026-11-10-19-42-26 +0000,nypost,Meghan Markle’s father Thomas cuts off ankle m...,https://nypost.com/2026/11/10/world-news/megha...
324,47,2026-11-10-10-24-30 +0000,nypost,US-mediated talks on ending Russia-Ukraine war...,https://nypost.com/2026/11/10/world-news/us-me...
115,42,2026-11-10-19-21-16 +0000,nypost,Nvidia loses nearly $450B in two-day wipeout a...,https://nypost.com/2026/11/10/business/nvidia-...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
